# 02 · Exhaustive sweep

Every matmul config on every shape (plus RMSNorm) is compiled, checked against an fp32 PyTorch
result and timed. This is the ground truth the predictor and the search strategies are judged
against. It runs 10 configs first to estimate the total time, then the rest. Rows are written to
disk as they are measured, so an interrupted run picks up where it stopped.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
DIMS = model.derive_dims(model.load_model_config(cache_dir=RESULTS))
MM_SHAPES = model.matmul_shapes(DIMS)
RMS_SHAPES = model.rmsnorm_shapes(DIMS)
SPLIT = {**model.train_test_split([s.shape_id for s in MM_SHAPES]),
         **model.train_test_split([s.shape_id for s in RMS_SHAPES])}
if MINI:
    small = model.mini_shapes(MM_SHAPES + RMS_SHAPES)
    MM_SHAPES = [s for s in small if s.kernel == "matmul"]
    RMS_SHAPES = [s for s in small if s.kernel == "rmsnorm"]
print(DIMS)
for s in MM_SHAPES + RMS_SHAPES:
    print(f"{s.shape_id:24s} {SPLIT[s.shape_id]:5s} {','.join(s.roles)}")

In [ ]:
import os
RUN_ID = config.EXPERIMENTS.sweep_run_id
MM_CFGS = model.mini_matmul_space() if MINI else model.matmul_space()
N_RMS = config.WORKLOAD.mini_rmsnorm_configs if MINI else None
TASKS = run.make_tasks(MM_SHAPES, lambda s: MM_CFGS) + run.make_tasks(RMS_SHAPES, lambda s: model.rmsnorm_space(s.N)[:N_RMS])
print(len(TASKS), "tasks:", len(MM_SHAPES), "matmul shapes x", len(MM_CFGS), "configs +", len(RMS_SHAPES), "rmsnorm shapes")

## First 10 configs and a time estimate

In [ ]:
small = [t for t in TASKS if t.shape.shape_id == MM_SHAPES[-1].shape_id][:config.EXPERIMENTS.estimate_sample]
run.run_sweep(small, RESULTS, ENV, run_id=RUN_ID)
done = run.load_parts(os.path.join(RESULTS, ENV["gpu_tag"], "sweep"))
sample = done[done.cfg_hash.isin([t.cfg_hash for t in small]) & (done.shape_id == small[0].shape.shape_id)]
remaining = run.pending(TASKS, run.keys_of(done), ENV["gpu_tag"], RUN_ID)
n_cold = len({(t.shape.kernel, t.cfg_hash) for t in remaining})
EST = run.estimate_seconds(sample, n_tasks=len(remaining), n_cold_compiles=n_cold)
print(sample[["cfg_hash", "status", "compile_time_s", "n_regs", "n_spills", "shared_bytes", "time_ms_median"]])
print(f"estimated: {EST['estimate_s'] / 3600:.2f} h for {len(remaining)} tasks ({n_cold} cold compiles)")
# These 10 configs fit on every GPU, so a failure here is a bug rather than data.
bad = sample[sample.status.isin(["compile_error", "launch_error", "incorrect"])]
assert len(sample) == len(small) and bad.empty, bad.error_msg.head(3).tolist()

## Full sweep

In [ ]:
run.run_sweep(TASKS, RESULTS, ENV, run_id=RUN_ID, estimate_s=EST["estimate_s"])
df = run.consolidate(os.path.join(RESULTS, ENV["gpu_tag"]))
print(df.groupby(["kernel", "status"]).size())
missing = run.pending(TASKS, run.keys_of(df), ENV["gpu_tag"], RUN_ID)
assert not missing, f"{len(missing)} tasks have no row"
assert df.status.isin(["compile_error", "launch_error"]).mean() < 0.02